In [1]:
import pandas as pd, numpy as np
from pathlib import Path
from unidecode import unidecode
import sys, os, csv
from datetime import datetime, time

src = Path.cwd().parent/"src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

pd.set_option('display.max_columns', None)

from routes.Paths import (
    Ruta_opl,
    Ruta_auxiliares,
    Ruta_clasificacion_ax,
    Ruta_festivos,
    Ruta_local,
    Ruta_global
)
from utils.Processing_OPL import data_opl
from utils.functions import load_files_from_folder

df_clasifica_aux = pd.read_csv(Ruta_clasificacion_ax, delimiter=";", encoding="latin-1")
df_clasifica_aux = df_clasifica_aux[["Atributo", "Aplica_Ocupacion", "Auxiliares"]]
df_clasifica_aux["Atributo"] = df_clasifica_aux["Atributo"].str.lower()
df_opl = data_opl(pd.read_excel(Ruta_opl, sheet_name="Combinado"))
df_opl["Correo_contratista"] = df_opl["Correo_contratista"].str.lower()

df_festivos = (
    pd.to_datetime( pd.read_csv(Ruta_festivos)["Fecha"], format= "%d/%m/%Y")
    .drop_duplicates()
    .values.astype("datetime64[D]")
)

require_columns = {
    "agent",
    "startinterval",
    "endinterval",
    "agent idle time",
    "agent incoming connecting time",
    "contacts missed",
    "agent on contact time",
    "agent outbound connecting time",
    "average agent api connecting time",
    "average agent callback connecting time",
    "average agent incoming connecting time"
}
delete_dup_subset = [
    "agent",
    "startinterval",
    "endinterval"
]

df_aux = load_files_from_folder(
    Ruta_auxiliares,
    require_columns,
    delete_dup_subset
)

In [2]:
columnas_datetime = [
        "startinterval"
        ]
for col in columnas_datetime:
    df_aux[col] = pd.to_datetime(df_aux[col].astype(str)
                   .str.rsplit("-", n=1)
                   .str[0], errors="coerce")

In [3]:
df_aux["Fecha"] = pd.to_datetime(df_aux["startinterval"].dt.date)
order_col = [
    "agent",
    "Fecha",
    "Intervalo_Ini",
    "source_file", 
    "file_date"
]
df_aux["Intervalo_Ini"] = df_aux["startinterval"].dt.time

column_order = order_col + [col for col in df_aux.columns if col not in order_col]
df_aux = df_aux[column_order]
df_aux.columns.to_list()
delete_columns = [
    "agent answer rate",
    "average agent api connecting time",
    "average agent callback connecting time",
    "average agent incoming connecting time",
    "average agent outbound connecting time",
    "online time",
    "source.name",
    "nonproductive time",
    "startinterval",
    "endinterval",
    "occupancy"
]
df_aux = df_aux.drop(columns=delete_columns, errors="ignore")

In [4]:
df_aux = pd.melt(
    df_aux,
    id_vars=order_col,
    var_name="Nombre_Estado",
    value_name="Time(seg)")
df_aux = df_aux[~df_aux["Time(seg)"].isna()]
df_aux["agent"] = df_aux["agent"].str.lower()

df_aux = df_aux.merge(
    df_opl[["Id_opl","Correo_contratista"]],
    how="left",
    left_on="agent",
    right_on="Correo_contratista",
).drop(columns=["Correo_contratista"])

inicio = time(8, 0, 0)
fin = time(20, 0, 0)

df_aux["Aplica_factura"] = np.where(
    (~df_aux["Fecha"].isin(df_festivos)) &
    (df_aux["Intervalo_Ini"] >= inicio) &
    (df_aux["Intervalo_Ini"] <= fin),
    "SI", "NO" 
)

In [5]:
df_aux = df_aux.merge(
    df_clasifica_aux[["Atributo","Aplica_Ocupacion", "Auxiliares"]],
    how="left",
    left_on="Nombre_Estado",
    right_on="Atributo"
).drop(columns=["Atributo"])

In [6]:
def guardar_data_auxiliares(
        df: pd.DataFrame,
        ruta_global: str,
        ruta_local: str,
        file_name: str = "06.Data_Conexion_Auxiliares.csv")-> None:


    df.to_csv(
        os.path.join(ruta_global, file_name),
        index=False,
        encoding="utf-8"
    )

    df.to_csv(
        os.path.join(ruta_local, file_name),
        index=False,
        encoding="utf-8"
    )

In [7]:
guardar_data_auxiliares(df_aux, Ruta_global, Ruta_local)